# BONITA: Generating Italian Instruction-Tuning Data for Boolean Question Answering

*Francesco Baiocchi, Leonardo Petrilli — Task 3, course project work*

**Goal.** BONITA is an Italian version of Bonito ([Nayak et al., 2024](https://arxiv.org/abs/2402.18334)): a model that turns unannotated text into instruction-tuning data. Given a passage and a task type, it generates a task, i.e. an instruction with its input, together with the expected answer. Here the task type is boolean question answering: from a passage, BONITA writes a yes/no question about it and the answer (`Vero`/`Falso`).

**Pipeline.**

1. **Train BONITA**: a QLoRA adapter (plus the embeddings of its special tokens) on [`sapienzanlp/Minerva-7B-base-v1.0`](https://huggingface.co/sapienzanlp/Minerva-7B-base-v1.0), using 80% of [`sapienzanlp/boolq_italian`](https://huggingface.co/datasets/sapienzanlp/boolq_italian) (`train` + `validation`, split by passage, see Section 2).
2. **Collect unannotated Italian text**: 10,000 paragraphs from Italian Wikipedia, the same kind of source as BoolQ, whose passages are English Wikipedia paragraphs translated into Italian. No label is attached to them.
3. **Generate a synthetic dataset**: BONITA writes one question/answer pair per paragraph. Generations that cannot be parsed and duplicate questions are discarded.
4. **Train the student**: a QLoRA adapter on [`sapienzanlp/Minerva-7B-instruct-v1.0`](https://huggingface.co/sapienzanlp/Minerva-7B-instruct-v1.0), trained on the synthetic dataset. We call it student because it learns the task only from data written by BONITA.
5. **Evaluate** the baseline model (Minerva-7B-instruct without the adapter) and the student on the remaining 20%, the test set (2,490 human-labeled examples). Test examples whose passage or question also appears in BONITA's data are removed, so the test set is never seen by BONITA. Each example is scored by log-likelihood, comparing `Vero` and `Falso`, with 3 instruction variants; we report accuracy and macro-F1 per variant and averaged, next to the majority baseline (always `Vero`).

**Contents**

- [1. Setup & configuration](#1-setup--configuration)
- [2. Data: BoolQ splits and BONITA meta-template](#2-data-boolq-splits-and-bonita-meta-template)
- [3. Training BONITA](#3-training-bonita)
- [4. Unannotated Italian text](#4-unannotated-italian-text)
- [5. Generating the instruction-tuning dataset with BONITA](#5-generating-the-instruction-tuning-dataset-with-bonita)
- [6. Training the student](#6-training-the-student)
- [7. Log-likelihood evaluation on the test set](#7-log-likelihood-evaluation-on-the-test-set)
- [8. Results](#8-results)
- [9. Error analysis](#9-error-analysis)
- [10. Conclusions](#10-conclusions)

## 1. Setup & configuration

The whole pipeline runs on a single NVIDIA GeForce RTX 3090 (24 GB). Models are handled with [`transformers`](https://github.com/huggingface/transformers) (loading, generation), [`bitsandbytes`](https://github.com/bitsandbytes-foundation/bitsandbytes) (4-bit quantization), [`peft`](https://github.com/huggingface/peft) (LoRA adapters) and [`trl`](https://github.com/huggingface/trl) (`SFTTrainer`, supervised fine-tuning on prompt/completion pairs); datasets are loaded from the Hugging Face Hub with [`datasets`](https://github.com/huggingface/datasets).

In [1]:
import os

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import gc
import json
import random
import re
import tempfile

import numpy as np
import pandas as pd
import torch
import transformers
from datasets import Dataset, concatenate_datasets, load_dataset
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, EarlyStoppingCallback
from trl.trainer.sft_config import SFTConfig
from trl.trainer.sft_trainer import SFTTrainer

In [2]:
SEED = 42

BASE_MODEL = "sapienzanlp/Minerva-7B-base-v1.0"
INSTRUCTED_MODEL = "sapienzanlp/Minerva-7B-instruct-v1.0"

# outputs of each stage (each stage can be re-run from its files)
BONITA_DIR = "ckpt/bonita_full"  # generator adapter + tokenizer
WIKI_PATH = "wiki_passages.jsonl"  # unannotated passages
GENERATED_PATH = "generated_tasks.jsonl"  # parsed and filtered BONITA tasks
STUDENT_DIR = "ckpt/student_bonita"  # student adapter
RESULTS_PATH = "results.json"  # generation statistics and metrics
PREDICTIONS_PATH = "predictions.jsonl"  # per-example test scores, for the error analysis

TEST_FRAC = 0.2  # share of BoolQ-Italian passages used as the test set of the baseline and the student
N_PASSAGES = 10_000  # unannotated paragraphs given to BONITA (~ size of BoolQ train)

# one of the 10 files of Italian Wikipedia (~180k articles) is enough to sample the passages
WIKI_SHARD = "hf://datasets/wikimedia/wikipedia/20231101.it/train-00000-of-00010.parquet"

# for both, the checkpoint with the lowest loss on a dev set made of 5% of the training passages is kept
# (evaluated every 100 steps, early stopping after 3 evaluations without improvement)
BONITA_EPOCHS = 4
STUDENT_EPOCHS = 4
DEV_FRAC = 0.05


def free_memory() -> None:
    gc.collect()
    torch.cuda.empty_cache()


def read_jsonl(path: str) -> list[dict]:
    with open(path) as f:
        return [json.loads(line) for line in f]


transformers.set_seed(SEED)  # seeds random, numpy and torch (CPU and CUDA)

The next cells define the components shared by all the models of the pipeline.

**Quantization.** Every model is loaded in 4-bit NF4, with double quantization and `bfloat16` compute.

**LoRA.** BONITA and the student are trained with the same LoRA configuration. Following QLoRA ([Dettmers et al., 2023](https://arxiv.org/abs/2305.14314)), which shows that adapting all linear layers is key to matching full fine-tuning performance, we apply LoRA to all linear projections, with dropout $0.05$. We use rank $r=16$ with $\alpha=32$ (i.e. $\alpha = 2r$). BONITA additionally trains the embedding and LM-head rows of its special tokens.

In [3]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)


def load_model(tokenizer=None, model_name: str = BASE_MODEL) -> transformers.PreTrainedModel:
    """4-bit model; the embeddings are resized only if `tokenizer` has no free rows left for its added tokens."""
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        dtype=torch.bfloat16,
    )
    if tokenizer is not None and len(tokenizer) > model.config.vocab_size:
        model.resize_token_embeddings(len(tokenizer))
    return model


def lora_config(trainable_token_ids: list[int] | None = None) -> LoraConfig:
    """LoRA on all linear projections, plus the rows of `trainable_token_ids` in the (untied) embeddings and LM head."""
    return LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules="all-linear",
        lora_dropout=0.05,
        task_type="CAUSAL_LM",
        trainable_token_indices={"embed_tokens": trainable_token_ids, "lm_head": trainable_token_ids}
        if trainable_token_ids
        else None,
    )

**Training.** Both models share the same arguments (`sft_config`). Following Bonito, we use an effective batch of 16, obtained with a batch of 8 and 2 gradient-accumulation steps. The learning rate is $5\times10^{-5}$ with cosine decay and 3% warmup; training uses `bfloat16` and lasts at most 4 epochs. 

As in Bonito, the loss is computed on the completion only: the prompt is read as input but is never a training target. For the student this matters most, since the target (` Vero`/` Falso`) is one or two tokens long while the passage has hundreds, and a loss on the whole sequence would be dominated by the passage. Since our training sets are small (about 10k examples), we do not fix the number of steps but select it on a dev set made of 5% of the training passages: every 100 steps the model is evaluated on the dev set, training stops early if the dev loss does not improve for 3 evaluations in a row, and the checkpoint with the lowest dev loss is restored at the end. This limits overfitting and lets the two models stop at different points with the same configuration.

In [4]:
def sft_config(output_dir: str, num_epochs: int) -> SFTConfig:
    """Training arguments shared by BONITA and the student; the checkpoint with the lowest dev loss is kept."""
    return SFTConfig(
        output_dir=output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        learning_rate=5e-5,
        lr_scheduler_type="cosine",
        warmup_steps=0.03,
        bf16=True,
        completion_only_loss=True,
        eval_strategy="steps",
        eval_steps=100,
        save_steps=100,
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        logging_steps=25,
        seed=SEED,
    )


def train_lora(
    model,
    tokenizer,
    train_rows: list[dict],
    dev_rows: list[dict],
    output_dir: str,
    num_epochs: int,
    trainable_token_ids: list[int] | None = None,
) -> None:
    """Adds a fresh LoRA to `model`, trains it, saves the best adapter to `output_dir` and prints the dev losses."""
    transformers.set_seed(SEED)  # the LoRA weights are initialized before the Trainer sets its own seed
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)  # freeze, fp32 norms, grad ckpt
    model = get_peft_model(model, lora_config(trainable_token_ids))
    assert isinstance(model, PeftModel)
    model.print_trainable_parameters()
    with tempfile.TemporaryDirectory() as ckpt_dir:  # intermediate checkpoints, deleted after training
        trainer = SFTTrainer(
            model=model,
            args=sft_config(ckpt_dir, num_epochs),
            train_dataset=Dataset.from_list(train_rows).select_columns(["prompt", "completion"]),
            eval_dataset=Dataset.from_list(dev_rows).select_columns(["prompt", "completion"]),
            processing_class=tokenizer,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],  # stop after 3 evals without improvement
        )
        trainer.train()
    # the best checkpoint is loaded into `model`; only the LoRA and the trained token rows are saved
    model.save_pretrained(output_dir, save_embedding_layers=False)
    tokenizer.save_pretrained(output_dir)
    print(pd.DataFrame(trainer.state.log_history).dropna(subset=["eval_loss"])[["step", "epoch", "eval_loss"]])

## 2. Data: BoolQ splits and BONITA meta-template

[BoolQ-Italian](https://huggingface.co/datasets/sapienzanlp/boolq_italian) is an Italian translation of [BoolQ](https://huggingface.co/datasets/google/boolq), a question-answering dataset composed of user queries issued to a search engine. The task is to predict whether the answer to the question is true or false based on the context provided in the question.

In [5]:
boolq = load_dataset("sapienzanlp/boolq_italian")
print(f"train examples: {len(boolq['train'])}")
print(f"validation examples: {len(boolq['validation'])}")

train examples: 9399
validation examples: 3259


### 2.1 Pre-processing

While manually inspecting the dataset, we noticed that some passages were duplicated across the splits. This comes from the original BoolQ, whose splits are drawn per question: different user queries, often paraphrases, were paired with the same Wikipedia paragraph.

To avoid data leakage we merge the two splits and divide them (80/20) by English passage, putting all the questions on a paragraph on the same side, so no passage is shared between BONITA's training data and the test set.

In [6]:
# passages that appear in both the original splits
train_passages = {m["passage_translation"] for m in boolq["train"]["metadata"]}
validation_passages = {m["passage_translation"] for m in boolq["validation"]["metadata"]}
print(f"passages in both train and validation: {len(train_passages & validation_passages)}")

passages in both train and validation: 487


In [7]:
def split_by_group(rows: list[dict], key: str, frac: float, seed: int) -> tuple[list[dict], list[dict]]:
    """Splits rows into (train, held-out) so that all rows sharing `key` end up on the same side."""
    groups = sorted({r[key] for r in rows})
    random.Random(seed).shuffle(groups)
    held_out = set(groups[: round(len(groups) * frac)])
    return [r for r in rows if r[key] not in held_out], [r for r in rows if r[key] in held_out]


boolq_all = [
    {**ex, "passage": ex["metadata"]["passage"]}
    for ex in concatenate_datasets([boolq["train"], boolq["validation"]]).to_list()
]

bonita_source, boolq_test = split_by_group(
    boolq_all, "passage", frac=TEST_FRAC, seed=SEED
)  # by original (English) passage
print(f"BONITA source: {len(bonita_source)}, test: {len(boolq_test)}")

BONITA source: 10094, test: 2564


The split by passage does not catch everything. Some English passages differ only by minor edits, likely because the same Wikipedia paragraph was collected in different revisions. These differences are lost in translation, so the two passages end up with the same Italian text. Likewise, the same question can be paired with different passages. The test examples whose Italian passage or normalized question also appears in BONITA's data are removed, so that the test set does not contain examples that BONITA has already seen in another form.

In [8]:
def normalize(question: str) -> str:
    return " ".join(re.findall(r"\w+", question.lower()))


# test examples whose Italian passage or normalized question also appear in BONITA's data
seen_passages = {ex["metadata"]["passage_translation"] for ex in bonita_source}
seen_questions = {normalize(ex["input_translation"]) for ex in bonita_source}


def is_near_duplicate(ex: dict) -> bool:
    passage, question = ex["metadata"]["passage_translation"], normalize(ex["input_translation"])
    return passage in seen_passages or question in seen_questions


print(f"near-duplicates of BONITA's data in the test set: {sum(is_near_duplicate(ex) for ex in boolq_test)}")

boolq_test = [ex for ex in boolq_test if not is_near_duplicate(ex)]

train_passages = {m["metadata"]["passage_translation"] for m in bonita_source}
test_passages = {m["metadata"]["passage_translation"] for m in boolq_test}
print(f"passages in both BONITA's data and the test set after filtering: {len(train_passages & test_passages)}")
print(f"near-duplicates after filtering: {sum(is_near_duplicate(ex) for ex in boolq_test)}")

near-duplicates of BONITA's data in the test set: 74
passages in both BONITA's data and the test set after filtering: 0
near-duplicates after filtering: 0


### 2.2 BONITA meta-template

Each example of BONITA's data becomes a prompt/completion pair:

**Prompt:**
```
<|tasktype|>
boolean question answering
<|context|>
{passage}
<|task|>
```

**Completion:**
```
{instruction} {question} Vero o Falso?
<|pipe|>
Risposta: Vero/Falso
```

The prompt contains the task type and the passage. The completion contains the task, that is the instruction followed by the question, and after `<|pipe|>` its answer. BONITA therefore learns to generate both the question and its answer from a passage. The instruction is drawn at random from three equivalent Italian wordings, so that the generator does not always produce the same one. A dev set made of 5% of BONITA's passages is used for early stopping and to select the best checkpoint.

In [9]:
INSTRUCTION_VARIANTS = [
    "Dopo aver letto il passaggio, rispondi alla seguente domanda:",
    "In base al testo precedente, rispondi alla domanda:",
    "Secondo quanto riportato nel passaggio, rispondi alla domanda:",
]
QUESTION_SUFFIX = "Vero o Falso?"
SPECIAL_TOKENS = ["<|tasktype|>", "<|context|>", "<|task|>", "<|pipe|>"]

BONITA_INPUT = "<|tasktype|>\nboolean question answering\n<|context|>\n{passage}\n<|task|>"
BONITA_OUTPUT = "\n{instruction} {question}\n<|pipe|>\n{answer}"
PROMPT_TEMPLATE = "{passage}\n\n{task}\nRisposta:"  # student training and evaluation (Sections 6-7)


def answer_text(label: bool) -> str:
    return "Vero" if label else "Falso"


rng = random.Random(SEED)
bonita_rows = [
    {
        "prompt": BONITA_INPUT.format(passage=ex["metadata"]["passage_translation"]),
        "completion": BONITA_OUTPUT.format(
            instruction=rng.choice(INSTRUCTION_VARIANTS),
            question=f"{ex['input_translation']} {QUESTION_SUFFIX}",
            answer=f"Risposta: {answer_text(ex['label'])}",
        ),
        "passage": ex["passage"],
    }
    for ex in bonita_source
]
bonita_train, bonita_dev = split_by_group(bonita_rows, "passage", frac=DEV_FRAC, seed=SEED)
print(f"BONITA train: {len(bonita_train)}, dev: {len(bonita_dev)}")
print(bonita_train[0]["prompt"] + bonita_train[0]["completion"])

BONITA train: 9611, dev: 483
<|tasktype|>
boolean question answering
<|context|>
Il persiano (/ˈpɜːrʒən, -ʃən/), conosciuto anche con il suo nome endonimo Farsi (فارسی fārsi (fɒːɾˈsiː) (ascolta)), è una delle lingue iraniche occidentali del ramo indo-iraniano della famiglia delle lingue indoeuropee. È parlato principalmente in Iran, Afghanistan (ufficialmente conosciuto come Dari dal 1958) e Tagikistan (ufficialmente conosciuto come Tajiki dall'era sovietica) e in alcune altre regioni che storicamente erano società persiane e considerate parte del Grande Iran. È scritto in alfabeto persiano, una variante modificata della scrittura araba, che a sua volta si è evoluta dall'alfabeto aramaico.
<|task|>
Secondo quanto riportato nel passaggio, rispondi alla domanda: L'Iran e l'Afghanistan parlano la stessa lingua? Vero o Falso?
<|pipe|>
Risposta: Vero


## 3. Training BONITA

The four markers of the meta-template (`<|tasktype|>`, `<|context|>`, `<|task|>`, `<|pipe|>`) are added to the tokenizer as special tokens, so that each one is a single, unsplittable token. The embedding matrix is resized only if it has no free rows left for them, which is the case for Minerva-7B-base (exactly 51,200 rows).

> **Note:** this section is **not meant to be re-run**. Training BONITA takes several GPU-hours: to train it again, use the script `scripts/train_bonita.py`. The run is documented here for transparency and as a reference. The trained adapter and tokenizer can be downloaded [here](https://drive.google.com/drive/folders/1fhtEb1as812R5gX4Ia1gRR245-ut1uoj) into `ckpt/bonita_full/`, from where Section 5 loads them: the next cell can be skipped.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.add_tokens(SPECIAL_TOKENS, special_tokens=True)
model = load_model(tokenizer)
special_token_ids = tokenizer.convert_tokens_to_ids(SPECIAL_TOKENS)
assert isinstance(special_token_ids, list)  # a list of tokens gives a list of ids

# LoRA + the 4 rows of the special tokens
train_lora(model, tokenizer, bonita_train, bonita_dev, BONITA_DIR, BONITA_EPOCHS, trainable_token_ids=special_token_ids)

del model
free_memory()

## 4. Unannotated Italian text

[BoolQ](https://arxiv.org/abs/1905.10044) passages are Wikipedia paragraphs, so the unannotated text is taken from Italian Wikipedia ([`wikimedia/wikipedia`](https://huggingface.co/datasets/wikimedia/wikipedia), `20231101.it`). Only the first of its 10 parquet files is downloaded (~180k articles, far more than needed) and its articles are shuffled. From each article we take one paragraph (ending with a full stop and at least as long as the 5th percentile of the length of BONITA's training passages in words, so that it is not much shorter than BoolQ passages), chosen at random.

The passages are collected only once and saved to `wiki_passages.jsonl`; if the file already exists it is loaded instead. To collect them again, delete the file. The passages used in our run can be downloaded [here](https://drive.google.com/file/d/1zeSdbuN2n-zZ2Nh5hLwcfHQTY8Y0JqyW/view).

In [10]:
MIN_WORDS = int(np.percentile([len(ex["metadata"]["passage_translation"].split()) for ex in bonita_source], 5))


def candidate_paragraphs(text: str) -> list[str]:
    paragraphs = (p.strip() for p in text.split("\n"))
    return [p for p in paragraphs if p.endswith(".") and len(p.split()) >= MIN_WORDS]


def collect_wiki_passages() -> list[dict]:
    wiki = load_dataset("parquet", data_files=WIKI_SHARD, split="train")
    assert isinstance(wiki, Dataset)
    wiki = wiki.shuffle(seed=SEED)
    rng = random.Random(SEED)
    passages = []
    for text in wiki["text"]:
        candidates = candidate_paragraphs(text)
        if candidates:
            passages.append({"passage": rng.choice(candidates)})
        if len(passages) >= N_PASSAGES:
            break
    return passages


# the passages are collected once
if os.path.exists(WIKI_PATH):
    print(f"loading the passages from {WIKI_PATH}")
    wiki_passages = read_jsonl(WIKI_PATH)
else:
    print(f"minimum paragraph length: {MIN_WORDS} words")
    wiki_passages = collect_wiki_passages()

print(f"{len(wiki_passages)} passages")
print(wiki_passages[0])

loading the passages from wiki_passages.jsonl
10000 passages
{'passage': "Uno dei più bei parchi del Nunavut, il Katannilik ('Luogo delle Cascate') si trova ad appena qualche decina di chilometri (percorribili a piedi, in aereo o motoslitta) da Iqaluit. Il parco vanta due grandi attrattive: il Soper River e l'itijjagiaq Trail. Il Super River è un fiume dalle acque color acquamarina dichiarato patrimonio culturale canadese, con un tratto navigabile di 50 chilometri che attraversa una profonda e fertile vallata e si snoda tra cascate, caribù, giacimenti di pietre preziose e una foresta di salici nani fino a raggiungere l'insediamento di Kimmirut. Di solito il percorso è frequentato da canoisti."}


## 5. Generating the instruction-tuning dataset with BONITA

BONITA generates one task per passage with greedy decoding (`do_sample=False`, for reproducibility); the other generation parameters are the defaults of Minerva-7B-base. Each generation is parsed into instruction + question and answer, and kept only if:

- it contains a parseable `Risposta: Vero/Falso` and the question ends with *"Vero o Falso?"*;
- the question is not a duplicate of an already kept question (after normalization).

The next cell reloads the base model with the BONITA adapter and generates in batches of 16 prompts, each made of the task type and a Wikipedia passage (the prompt part of the meta-template). The prompts of a batch have different lengths and are padded **on the left**: in this way every prompt ends right where generation starts, and the new tokens can be taken as everything after the input.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BONITA_DIR)
model = PeftModel.from_pretrained(load_model(tokenizer), BONITA_DIR)
model.eval()


@torch.no_grad()
def batch_generate(model, tokenizer, prompts: list[str], batch_size: int = 16, **generation_kwargs) -> list[str]:
    """Generates one completion per prompt (new tokens only, special tokens removed)."""
    predictions = []
    for i in tqdm(range(0, len(prompts), batch_size)):
        # left padding: every prompt ends right before the new tokens
        inputs = tokenizer(prompts[i : i + batch_size], padding=True, padding_side="left", return_tensors="pt").to(
            model.device
        )
        outputs = model.generate(
            **inputs, pad_token_id=tokenizer.pad_token_id, tokenizer=tokenizer, **generation_kwargs
        )
        new_tokens = outputs[:, inputs["input_ids"].shape[1] :]
        predictions += tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
    return predictions


generations = batch_generate(
    model,
    tokenizer,
    [BONITA_INPUT.format(passage=p["passage"]) for p in wiki_passages],
    max_new_tokens=256,
    stop_strings=["Risposta: Vero", "Risposta: Falso"],
    do_sample=False,
    use_cache=True,  # Minerva's config disables the KV cache
)

del model
free_memory()

**Parsing and filtering.** A generation is expected to look like `{instruction} {question} Vero o Falso? Risposta: Vero/Falso` (the `<|pipe|>` token is removed when decoding). The label is found with a regular expression on `Risposta: Vero/Falso`; everything before it is the *task* (instruction + question), which is what the student will see. The bare question (task without instruction and suffix) is only used for deduplication, after lowercasing and removing punctuation.

The kept tasks (passage, task, question, label) are saved to `generated_tasks.jsonl`. The counts of malformed and duplicate generations and the share of `Vero` labels, which shows whether BONITA's answers are balanced, are saved to `results.json` in Section 8.

In [ ]:
LABEL_RE = re.compile(r"Risposta:\s*(Vero|Falso)\b", re.IGNORECASE)


def parse_generation(text: str) -> dict | None:
    """Splits a BONITA generation into task (instruction + question), question and label."""
    match = LABEL_RE.search(text)
    if match is None:
        return None
    task = text[: match.start()].strip()
    if not task.endswith(QUESTION_SUFFIX):
        return None
    question = task.removesuffix(QUESTION_SUFFIX)
    for variant in INSTRUCTION_VARIANTS:
        question = question.replace(variant, "")
    return {"task": task, "question": question.strip(), "label": match.group(1).capitalize() == "Vero"}


generated, seen = [], set()
counts: dict[str, float] = {"generated": len(generations), "malformed": 0, "duplicate": 0}
for passage, text in zip(wiki_passages, generations, strict=True):
    parsed = parse_generation(text)
    if parsed is None:
        counts["malformed"] += 1
        continue
    key = normalize(parsed["question"])
    if key in seen:
        counts["duplicate"] += 1
        continue
    seen.add(key)
    generated.append({"passage": passage["passage"], **parsed})


counts["kept"] = len(generated)
counts["vero_rate"] = sum(r["label"] for r in generated) / len(generated) if generated else 0.0
print(counts)
print(generated[0]["passage"], "...")
print("  ->", generated[0]["task"], "|", answer_text(generated[0]["label"]))

## 6. Training the student

The student is a fresh LoRA on Minerva-7B-instruct, trained on the generated tasks only. Each task becomes a prompt/completion pair in the same format used for evaluation in Section 7:

```
{passage}

{instruction} {question} Vero o Falso?
Risposta:
```

with completion ` Vero` or ` Falso`. The loss is computed on the completion only. A dev set made of 5% of the training passages is used for early stopping and to select the best checkpoint.

The code cell below the note reloads the generated tasks from `generated_tasks.jsonl` if the file exists, so that Sections 3-5 can be skipped. The tasks generated in our run can be downloaded [here](https://drive.google.com/file/d/1DFTBdmEZUq34UNDmzeAHe8hkFrIZKmnf/view). It then splits the rows by passage, loads a new 4-bit Minerva-7B-instruct, adds a LoRA and trains it with the shared arguments of Section 1.

> **Note:** this section is **not meant to be re-run**. Training the student takes several GPU-hours: to train it again, use the script `scripts/train_student.py`. The run is documented here for transparency and as a reference. The trained adapter can be downloaded [here](https://drive.google.com/drive/folders/11j1bIspr6NBVIWKXAmmUs1GM1KvwTBOD) into `ckpt/student_bonita/`, from where Section 7 loads it: the next cell can be skipped.

In [ ]:
# the generated tasks are reloaded from disk if they exist, so Sections 3-5 can be skipped
if os.path.exists(GENERATED_PATH):
    print(f"loading the generated tasks from {GENERATED_PATH}")
    generated = read_jsonl(GENERATED_PATH)

student_rows = [
    {
        "prompt": PROMPT_TEMPLATE.format(passage=r["passage"], task=r["task"]),
        "completion": " " + answer_text(r["label"]),
        "passage": r["passage"],
    }
    for r in generated
]
student_train, student_dev = split_by_group(student_rows, "passage", frac=DEV_FRAC, seed=SEED)
print(f"student train: {len(student_train)}, dev: {len(student_dev)}")
print(student_train[0]["prompt"] + student_train[0]["completion"])

tokenizer = AutoTokenizer.from_pretrained(INSTRUCTED_MODEL)
train_lora(load_model(model_name=INSTRUCTED_MODEL), tokenizer, student_train, student_dev, STUDENT_DIR, STUDENT_EPOCHS)
free_memory()

## 7. Log-likelihood evaluation on the test set

Every test example is turned into the prompt of Section 6, once for each of the three instruction variants, for example with the first one:

```
{passage}

Dopo aver letto il passaggio, rispondi alla seguente domanda: {question} Vero o Falso?
Risposta:
```

and the model scores the two continuations ` Vero` and ` Falso`: $\log P(\text{continuation} \mid \text{prompt})$ is the sum of the log-probabilities of the continuation tokens. A continuation can span several tokens (` Vero` is one token, ` Falso` two: `ĠFal` + `so`), and the probability of each token after the first depends on the previous ones: each (prompt, continuation) pair is therefore scored with its own forward pass, i.e. two forward passes per example, one for each continuation. The answer is the continuation with the highest score. We report the accuracy and the macro-F1, since the classes are unbalanced (61.0% `Vero`). The baseline that always answers `Vero` (majority class) is shown for reference.

In [11]:
CHOICES = [" Falso", " Vero"]

# one test set per instruction variant: same questions and labels, different instruction
eval_prompts = [
    [
        PROMPT_TEMPLATE.format(
            passage=ex["metadata"]["passage_translation"],
            task=f"{instruction} {ex['input_translation']} {QUESTION_SUFFIX}",
        )
        for ex in boolq_test
    ]
    for instruction in INSTRUCTION_VARIANTS
]
gold_labels = np.array([int(ex["label"]) for ex in boolq_test])
print(len(gold_labels), "test examples,", len(eval_prompts), "instruction variants")
print(eval_prompts[0][0])


def encode_request(tokenizer, context: str, continuation: str) -> tuple[list[int], int]:
    """Returns the token ids of context + continuation and the number of context tokens."""
    context_ids = tokenizer(context).input_ids
    return context_ids + tokenizer(continuation, add_special_tokens=False).input_ids, len(context_ids)


@torch.no_grad()
def continuation_logprobs(
    model, tokenizer, contexts: list[str], continuations: list[str], batch_size: int = 16
) -> np.ndarray:
    """Sum of the log-probabilities of each continuation given its context."""
    requests = [encode_request(tokenizer, c, x) for c, x in zip(contexts, continuations, strict=True)]
    scores = []
    for start in tqdm(range(0, len(requests), batch_size)):
        batch = requests[start : start + batch_size]
        # right padding: the positions of the real tokens are unchanged
        inputs = tokenizer.pad({"input_ids": [ids for ids, _ in batch]}, padding_side="right", return_tensors="pt")
        logits = model(**inputs.to(model.device)).logits
        for row, (ids, n_context) in enumerate(batch):
            # the token at position t is predicted by the logits at position t - 1
            logprobs = torch.log_softmax(logits[row, n_context - 1 : len(ids) - 1].float(), dim=-1)
            targets = torch.tensor(ids[n_context:], device=logprobs.device)
            scores.append(logprobs.gather(-1, targets[:, None]).sum().item())
    return np.array(scores)


def evaluate_model(model, tokenizer, prompts: list[str]) -> np.ndarray:
    """Log-likelihoods of both continuations for every prompt, shape (n, 2); the argmax is the predicted label."""
    contexts = [p for p in prompts for _ in CHOICES]
    continuations = [c for _ in prompts for c in CHOICES]
    return continuation_logprobs(model, tokenizer, contexts, continuations).reshape(-1, len(CHOICES))

2490 test examples, 3 instruction variants
Now You See Me è una serie di film di genere thriller e colpi scritti da Ed Solomon, Boaz Yakin e Edward Ricourt. I film si concentrano sulle azioni di un team di illusionisti chiamati ``The Four Horsemen'' che portano a termine colpi quasi impossibili. La serie presenta un cast di attori tra cui Jesse Eisenberg, Mark Ruffalo, Woody Harrelson, Isla Fisher, Dave Franco, Michael Caine, Lizzy Caplan e Morgan Freeman. Il primo film è stato rilasciato nel 2013, mentre il secondo è stato rilasciato nel 2016 e un terzo film è attualmente in fase di sviluppo e dovrebbe essere rilasciato nel 2019. La serie ha ricevuto recensioni miste da critici e pubblico e ha incassato quasi 700 milioni di dollari in tutto il mondo.

Dopo aver letto il passaggio, rispondi alla seguente domanda: Ci sarà un terzo capitolo di Now You See Me? Vero o Falso?
Risposta:


Minerva-7B-instruct is loaded once in 4-bit with the student's adapter: the student is scored with the adapter on, the baseline with the adapter turned off.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(INSTRUCTED_MODEL)
model = PeftModel.from_pretrained(load_model(model_name=INSTRUCTED_MODEL), STUDENT_DIR)
model.eval()

# one array of log-likelihoods [Falso, Vero] per instruction variant
scores: dict[str, list[np.ndarray]] = {"Baseline": [], "Student": []}
for prompts in eval_prompts:
    with model.disable_adapter():
        scores["Baseline"].append(evaluate_model(model, tokenizer, prompts))
    scores["Student"].append(evaluate_model(model, tokenizer, prompts))
predictions = {name: [s.argmax(axis=1) for s in per_variant] for name, per_variant in scores.items()}

## 8. Results

For each model, accuracy and macro-F1 are computed from the predictions of Section 7. The majority baseline answers `Vero` to every question: its accuracy is the share of `Vero` in the test set, while its macro-F1 is low, because the `Falso` class is never predicted.

In [12]:
def macro_f1(pred: np.ndarray, gold: np.ndarray) -> float:
    f1s = []
    for c in (0, 1):
        tp = np.sum((pred == c) & (gold == c))
        fp = np.sum((pred == c) & (gold != c))
        fn = np.sum((pred != c) & (gold == c))
        f1s.append(2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 0.0)
    return float(np.mean(f1s))


def summarize(pred: np.ndarray) -> dict[str, float]:
    return {"acc": float(np.mean(pred == gold_labels)), "macro_f1": macro_f1(pred, gold_labels)}


if "predictions" not in globals() and os.path.exists(RESULTS_PATH):
    # Section 7 was skipped: the metrics saved by a previous run are reloaded
    with open(RESULTS_PATH) as f:
        results = json.load(f)["results"]
else:
    results = {"always Vero (majority)": summarize(np.ones_like(gold_labels))}
    for name, preds in predictions.items():
        per_variant = [summarize(pred) for pred in preds]
        for i, metrics in enumerate(per_variant):
            results[f"{name}, instruction {i}"] = metrics
        results[f"{name}, mean"] = {k: float(np.mean([s[k] for s in per_variant])) for k in per_variant[0]}

# same layout as the table of Section 9.1 and of the report
table = {}
for key, metrics in results.items():
    model, _, instruction = key.partition(", ")  # e.g. "Baseline, instruction 0"
    if not instruction:  # the majority baseline
        model, instruction = "Always Vero", "-"
    table[model, instruction.removeprefix("instruction ")] = metrics
n_rows = len(results) // 2  # rows of Baseline and Student, mean included
(100 * pd.DataFrame(table).T).rename(columns={"acc": "Accuracy", "macro_f1": "Macro-F1"}).rename_axis(
    ["Model", "Instr."]
).style.format("{:.1f}").set_table_styles(
    [
        {"selector": "th.level0", "props": "vertical-align: middle; text-align: center"},
        # lines before Baseline and before Student
        {"selector": "tbody tr:nth-child(2) > *", "props": "border-top: 1px solid"},
        {"selector": f"tbody tr:nth-child({n_rows + 2}) > *", "props": "border-top: 1px solid"},
    ]
)

## 9. Error analysis

This section looks at where the gain of the student in Section 8 comes from and at the errors that the two models still make. It reads the per-example log-likelihoods saved by Section 7 to `predictions.jsonl`: for each test example, the file stores the passage, the Italian question, the original English question of BoolQ, the gold label and, for each model and each of the three instruction variants, the log-likelihoods of the two continuations ` Falso` and ` Vero`.

We summarize each pair of log-likelihoods with the **margin** $\log P(\text{Vero}) - \log P(\text{Falso})$: the model predicts `Vero` when the margin is positive, and the absolute value of the margin measures how confident the prediction is.

In [13]:
MODELS = ["Baseline", "Student"]

test_predictions = read_jsonl(PREDICTIONS_PATH)  # saved in Section 7, so that the evaluation can be skipped
gold = np.array([int(r["label"]) for r in test_predictions])
# log-likelihoods [Falso, Vero], shape (examples, instructions, 2)
log_likelihoods = {m: np.array([r["scores"][m] for r in test_predictions]) for m in MODELS}
margin = {m: ll[..., 1] - ll[..., 0] for m, ll in log_likelihoods.items()}  # (examples, instructions), > 0 means Vero
correct = {m: (margin[m] > 0) == gold[:, None] for m in MODELS}

print(f"{len(test_predictions)} test examples, {margin['Student'].shape[1]} instructions")
print(f"share of Vero in the test set: {100 * gold.mean():.1f}%")

2490 test examples, 3 instructions
share of Vero in the test set: 61.0%


### 9.1 Predictions per class

We look at the two classes separately to see whether the gain of the student comes from both answers or from only one of them. For each model and instruction, the table shows how often the model answers `Vero` and its recall on each class.

In [14]:
def class_metrics(pred: np.ndarray) -> dict[str, float]:
    """Share of Vero answers and recall of each class."""
    return {
        "Pred. Vero": pred.mean(),
        "Recall Vero": pred[gold == 1].mean(),
        "Recall Falso": 1 - pred[gold == 0].mean(),
    }


rows = {}
for m in MODELS:
    for i, pred in enumerate((margin[m] > 0).T):
        rows[m, str(i)] = class_metrics(pred)
    rows[m, "mean"] = class_metrics(margin[m] > 0)
(100 * pd.DataFrame(rows).T).rename_axis(["Model", "Instr."]).style.format("{:.1f}").set_table_styles(
    [
        {"selector": "th.level0", "props": "vertical-align: middle; text-align: center"},
        # line between Baseline and Student, as in the report
        {"selector": f"tbody tr:nth-child({margin['Baseline'].shape[1] + 2}) > *", "props": "border-top: 1px solid"},
    ]
)

The gain of the student comes from the `Vero` examples. Its recall on `Vero` rises from 74.6 to 88.2, while the recall on `Falso` barely changes, from 88.8 to 87.7. The baseline answers `Vero` on only 49.9% of the test examples and misses many of the `Vero` ones; the student answers `Vero` more often, on 58.6% of them.

### 9.2 Sensitivity to the instruction

The three instructions ask the same question with different wordings, so a robust model should give the same answer with all of them. This analysis checks whether the answers of the two models depend on how the question is introduced.

For each model, the table reports the share of test examples answered in the same way with the three instructions, and how many examples are answered correctly with all, none or only some of them, in which case the answer depends on the wording.

In [15]:
rows = []
for m in MODELS:
    n_correct = correct[m].sum(1)
    rows.append(
        {
            "model": m,
            "same answer for all instr (%)": round(100 * (n_correct % 3 == 0).mean(), 1),
            "right for all instr": (n_correct == 3).sum(),
            "wrong for all instr": (n_correct == 0).sum(),
            "right for some instr": ((n_correct > 0) & (n_correct < 3)).sum(),
        }
    )
pd.DataFrame(rows)

,model,same answer for all instr (%),right for all instr,wrong for all instr,right for some instr
0,Baseline,91.9,1890,399,201
1,Student,97.3,2157,265,68


The student is more robust to the wording of the instruction: it gives the same answer with all three instructions on 97.3% of the test examples, against 91.9% for the baseline, and it is right only with some of them on 68 examples, against 201.

To see how the examples move from one model to the other, the next table crosses the groups of the baseline, on the rows, with those of the student, on the columns. Its totals are the counts of the previous table.

In [16]:
GROUPS = ["wrong for all", "right for some", "right for all"]


def instruction_group(c: np.ndarray) -> "pd.Categorical[str]":
    """Whether a model is right with none, some or all of the three instructions."""
    n_correct = c.sum(1)
    return pd.Categorical(np.select([n_correct == 0, n_correct == 3], [GROUPS[0], GROUPS[2]], GROUPS[1]), GROUPS)


pd.crosstab(
    instruction_group(correct["Baseline"]),
    instruction_group(correct["Student"]),
    rownames=["Baseline"],
    colnames=["Student"],
    margins=True,
    margins_name="total",
)

Student,wrong for all,right for some,right for all,total
Baseline,,,,
wrong for all,176,25,198,399
right for some,33,9,159,201
right for all,56,34,1800,1890
total,265,68,2157,2490


Of the 201 examples on which the baseline depends on the wording, the student answers 159 correctly with all three instructions.

Overall the student is right with all three instructions on 2,157 examples, 267 more than the baseline. They come in almost equal parts from examples the baseline got wrong with all three instructions, 399 against 265, and from examples it got right only with some of them, 201 against 68.

### 9.3 Moving the decision threshold

Section 9.1 shows that the baseline makes more errors on the `Vero` examples than on the `Falso` ones. Part of the gain of the student could therefore come simply from answering `Vero` more often. The recalls alone cannot tell how much. If the baseline misses the `Vero` examples by a small margin, a slightly lower threshold recovers most of them and the baseline gets close to the student. If instead it gives the wrong answer with a large margin, moving the threshold recovers few of them and turns some correct `Falso` answers into errors. To measure it, we move the decision threshold on the margin of each model from 0 to the constant value that maximizes its accuracy.

> **Note:** since the threshold is chosen on the test set itself, the accuracy it gives is **not a fair score of the model**: it is an upper bound of what changing how often a model answers `Vero` can give.

In [17]:
thresholds = np.linspace(-5, 5, 201)
rows = []
for m in MODELS:
    accs = [((margin[m] > t) == gold[:, None]).mean() for t in thresholds]
    best = int(np.argmax(accs))
    rows.append(
        {
            "model": m,
            "mean margin": margin[m].mean(),
            "acc, threshold 0 (%)": round(100 * correct[m].mean(), 1),
            "best threshold": thresholds[best],
            "acc, best threshold (%)": round(100 * accs[best], 1),
        }
    )
pd.DataFrame(rows)

,model,mean margin,"acc, threshold 0 (%)",best threshold,"acc, best threshold (%)"
0,Baseline,-0.178607,80.1,-0.60,82.4
1,Student,0.551351,88.0,-0.35,88.3


With the best threshold the baseline goes from 80.1 to 82.4, still well below the 88.0 of the student, while the student, whose recalls on the two classes are already similar, barely changes. Answering `Vero` more often therefore accounts for about 2 of the 8 points gained by the student.

### 9.4 Examples

Reading single examples helps to understand which errors are behind the numbers of the previous sections and to spot problems in the data.

We take the examples on which both models give the same answer with all three instructions, and split them into three groups: those **fixed** by the student, wrong for the baseline and right for the student; those **broken** by the student, right for the baseline and wrong for the student; and those wrong for both. For each group we print five examples, chosen as explained in its subsection.

Each example starts with a line such as `[train-03558] gold: Falso, Baseline margin: -1.58, Student margin: +4.00`, which contains:

- the id of the example in BoolQ-Italian and its gold label;
- for each model, `gold_margin`: the margin of Section 9 averaged over the three instructions and signed towards the gold label. It is positive when the model gives the correct answer and negative when it gives the wrong one, and the larger its absolute value, the more strongly the model prefers that answer.

In this line the correct answer is `Falso`: the baseline answers `Vero`, while the student answers `Falso` with a large margin.

In [18]:
# mean margin over the instructions, > 0 when the model leans towards the gold label
gold_margin = {m: margin[m].mean(1) * np.where(gold == 1, 1, -1) for m in MODELS}


def show_examples(idx: np.ndarray, n: int = 5) -> None:
    for i in idx[:n]:
        r = test_predictions[i]
        print(f"[{r['id']}] gold: {'Vero' if r['label'] else 'Falso'}", end="")
        print("".join(f", {m} margin: {gold_margin[m][i]:+.2f}" for m in MODELS))
        print(f"  Q:  {r['question']}\n  EN: {r['question_en']}\n  P:  {r['passage']}\n")


right = {m: correct[m].all(1) for m in MODELS}  # right with all three instructions
wrong = {m: ~correct[m].any(1) for m in MODELS}  # wrong with all three instructions
fixed = np.where(wrong["Baseline"] & right["Student"])[0]
broken = np.where(right["Baseline"] & wrong["Student"])[0]
both_wrong = np.where(wrong["Baseline"] & wrong["Student"])[0]
gain = gold_margin["Student"] - gold_margin["Baseline"]

#### Fixed by the student

The fixed examples on which the margin towards the gold label grows the most from the baseline to the student.

In [19]:
show_examples(fixed[np.argsort(-gain[fixed])])

[validation-02125] gold: Vero, Baseline margin: -4.08, Student margin: +2.00
  Q:  La lucciola è la stessa cosa di una lanterna volante?
  EN: Is a lightning bug the same as a firefly?
  P:  I Lampyridae sono una famiglia di insetti nell'ordine dei coleotteri. Sono coleotteri alati, comunemente chiamati lucciole per il loro uso evidente di bioluminescenza durante il crepuscolo per attirare partner o prede. Le lucciole producono una "luce fredda", senza frequenze infrarosse o ultraviolette. Questa luce prodotta chimicamente dal basso addome può essere gialla, verde o rossa pallida, con lunghezze d'onda da 510 a 670 nanometri. Negli Stati Uniti orientali vive la specie Phausis reticulata, che emette una luce blu costante.

[train-08614] gold: Vero, Baseline margin: -1.75, Student margin: +4.21
  Q:  Fare irruzione in casa è la stessa cosa di furto con scasso?
  EN: Is breaking and entering the same as burglary?
  P:  Il furto, chiamato anche scasso e talvolta effrazione, è un ingresso il

#### Broken by the student

The broken examples on which the margin towards the gold label falls the most from the baseline to the student.

In [20]:
show_examples(broken[np.argsort(gain[broken])])

[validation-01363] gold: Falso, Baseline margin: +2.83, Student margin: -1.92
  Q:  Un motore a reazione è un motore a combustione esterna?
  EN: Is a jet engine an external combustion engine?
  P:  Nel linguaggio comune, il termine motore a reazione si riferisce in modo generico a un motore a reazione ad aria respirata a combustione interna. Questi motori sono caratterizzati da un compressore d'aria rotante alimentato da una turbina, con la potenza residua che fornisce la spinta tramite un ugello propulsore: questo processo è noto come ciclo termodinamico di Brayton. Gli aerei a reazione utilizzano questi motori per i viaggi a lunga distanza. I primi aerei a reazione utilizzavano motori a getto relativamente inefficienti per il volo subsonico. Gli aerei a reazione subsonici moderni utilizzano in genere motori turboelica ad alto bypass più complessi. Questi motori offrono una velocità elevata e una maggiore efficienza nei consumi di carburante rispetto ai motori a pistoni e a elica su 

#### Wrong for both models

The examples that both models get wrong, starting from those on which the student is most confidently wrong, that is with the most negative margin. Comparing the Italian question with the English one and with the passage shows whether an error comes from the model, from the translation, or the label of the example.

In [21]:
show_examples(both_wrong[np.argsort(gold_margin["Student"][both_wrong])])

[validation-01842] gold: Vero, Baseline margin: -4.46, Student margin: -6.75
  Q:  Il carbonio è un metallo nel sistema periodico?
  EN: Is carbon a metal on the periodic table?
  P:  Il carbonio (dal latino: carbo, "carbone") è un elemento chimico con simbolo C e numero atomico 6. È un elemento non metallico e tetravalente, ovvero dispone di quattro elettroni disponibili a formare legami chimici covalenti. Appartiene al gruppo 14 della tavola periodica. In natura si trovano tre isotopi, C e C sono stabili, mentre C è un radionuclide con un'emivita di circa 5.730 anni. Il carbonio è uno dei pochi elementi noti fin dall'antichità.

[validation-01841] gold: Falso, Baseline margin: -2.38, Student margin: -6.54
  Q:  La storia della tartaruga e della lepre è una favola?
  EN: Is the tortoise and the hare a fairy tale?
  P:  La Tartaruga e la Lepre" è una delle favole di Esopo e la numero 226 nell'Indice di Perry. Il racconto di una corsa tra partner diseguali ha attirato interpretazioni co

Some of these errors come from the data rather than from the models. In the first example above the gold label is `Vero`, while the passage says that carbon is not a metal. In the third example the English question asks whether the medieval period and the Middle Ages are the same, while its Italian translation asks about the Middle Ages and the modern age. Inspecting more errors by hand reveals other cases of the same kind, also outside this group: in the first fixed example, the English "lightning bug", a synonym of firefly, is translated as "lanterna volante", a paper sky lantern. For the Italian question the answer of the baseline, `Falso`, is the correct one, and the student is counted as right only because the gold label comes from the English question.

## 10. Conclusions

The project shows that the idea of Bonito carries over to Italian: starting from a base model and a translated dataset, BONITA turns plain Wikipedia text into instruction-tuning data from which a model can learn the task. The student never sees a human label, yet it improves over Minerva-7B-instruct on the test set, from 80.1 to 88.0 mean accuracy, and gives more consistent answers when the instruction is worded differently. The error analysis shows that the gain is not only a shift towards answering `Vero` more often. In practice, this means that once BONITA is trained, a new collection of Italian text can be turned into training data for boolean question answering at the cost of a generation run, with no human annotation.

**Limitations.**

- The student never sees BoolQ, but BONITA learned from BoolQ-Italian how to write its questions, and it applies them to Italian Wikipedia, the same kind of source as the BoolQ passages. The synthetic data is therefore likely to be close to the test set, and the results do not tell how well the pipeline works on text from a different domain, which is the main use case of Bonito.
- We compared the student only with Minerva-7B-instruct without the adapter, so we do not know where its gain comes from. Part of it may come simply from training on more Italian Wikipedia text, or from learning to answer with `Vero` or `Falso`, rather than from the questions and labels written by BONITA. A model trained with next-word prediction on the same paragraphs would measure the effect of the text alone, and a student trained on the human-labeled data of BoolQ-Italian would show how far the synthetic data is from human labels.
- BONITA covers a single task type, boolean question answering, while Bonito generates many.
- BoolQ-Italian is a machine translation, and Section 9.4 shows that some test examples have wrong labels or translations, so some of the errors counted on the test set are not errors of the models.

**Possible extensions.**

- Apply BONITA to unannotated text from a specialized Italian domain, such as legal or medical documents, and evaluate the student on a test set from that domain.
- Train a model with next-word prediction on the same Wikipedia paragraphs and a student on the human-labeled data of BoolQ-Italian, to measure how much of the gain comes from the questions and labels written by BONITA.
- Train BONITA on more task types, for example extractive question answering or natural language inference, using other Italian datasets.
- Clean the test set of the examples with wrong labels or translations, to get a more reliable measure of the error rate.